In [1]:
import numpy as np
import datetime
from train import train_model as tm
from predictor import predictor as pr
from cell_tracking import tracker as ct
import pandas as pd
from convert2CTCFormat import lineage_mask
from os.path import join
from multiprocessing import cpu_count
import hydra
from hydra.utils import to_absolute_path as abs_path
from hydra import compose, initialize
from omegaconf import OmegaConf

## Learn representation of cell movement

In [2]:
now = datetime.datetime.now()
print('START!', now)
with initialize(version_base='1.3', config_path="config"):
    cfg = compose(config_name="tracker")
    tm(cfg)
    now = datetime.datetime.now()
    print('train DONE!', now)

START! 2026-09-07 09:21:13.177820


INFO: Using device cuda


please check the configures:

path: ''
dataloader:
  division_detect: true
  start_frame: 0
  num_frame: 200
  itv: 1
  if_crop: false
  tile_num: 25
  pred_tile_num: 4
  overlap: 32
train:
  epochs: 100
  batch_size: 2
  lr: 0.0001
  train_load: false
  load: CP.pth
  val: false
track:
  max_movenment: 50
  load_track: false
  track_file: track_linear_solver.csv
  centroid_file: 2025-09-22-centroid.npy
  method: linear_solver
  run_num: 1
  last_itv: 20
  division: true
  post_pro: true
  min_length: 1
  prune_leaf: true
  merge: true
  jitter_thr: 0.6
  div_interval: 30
  nearest: false



INFO: Creating dataset with 5 examples
INFO: Starting training:
        Epochs:          100
        Batch size:      2
        Learning rate:   0.0001
        Training size:   4
        Checkpoints:     result\checkpoint
        Device:          cuda
        Interval:        1
        Optimizer:       Adam
    
Epoch 1/100: 100%|████████████████████████████████████████████████| 4/4 [00:17<00:00,  4.34s/img, loss (batch)=1.77e+3]
INFO: Checkpoint 1 saved !
Epoch 2/100: 100%|████████████████████████████████████████████████| 4/4 [00:30<00:00,  7.73s/img, loss (batch)=1.31e+3]
INFO: Checkpoint 2 saved !
Epoch 3/100: 100%|████████████████████████████████████████████████████| 4/4 [00:16<00:00,  4.24s/img, loss (batch)=881]
INFO: Checkpoint 3 saved !
Epoch 4/100: 100%|████████████████████████████████████████████████████| 4/4 [00:16<00:00,  4.07s/img, loss (batch)=618]
INFO: Checkpoint 4 saved !
Epoch 5/100: 100%|████████████████████████████████████████████████████| 4/4 [00:16<00:00,  4.10s/i

train DONE! 2026-09-07 09:47:32.761905


## predict movement field

In [3]:
now = datetime.datetime.now()
print('START!', now)
with initialize(version_base='1.3', config_path="config"):
    cfg = compose(config_name="tracker")
    pr(cfg)
    now = datetime.datetime.now()
    print('predict DONE!', now)

START! 2026-09-07 09:47:32.776382
please check the configures:

path: ''
dataloader:
  division_detect: true
  start_frame: 0
  num_frame: 200
  itv: 1
  if_crop: false
  tile_num: 25
  pred_tile_num: 4
  overlap: 32
train:
  epochs: 100
  batch_size: 2
  lr: 0.0001
  train_load: false
  load: CP.pth
  val: false
track:
  max_movenment: 50
  load_track: false
  track_file: track_linear_solver.csv
  centroid_file: 2025-09-22-centroid.npy
  method: linear_solver
  run_num: 1
  last_itv: 20
  division: true
  post_pro: true
  min_length: 1
  prune_leaf: true
  merge: true
  jitter_thr: 0.6
  div_interval: 30
  nearest: false



INFO: Using device cuda
INFO: Model loaded from result\checkpoint\CP_epoch99.pth
INFO: Creating dataset with 5 examples
                                                                                                                       

predict DONE! 2026-09-07 09:47:39.263387


## Bayesian Estimation and cell tracking

In [4]:
now = datetime.datetime.now()
print('START!', now)
with initialize(version_base='1.3', config_path="config"):
    cfg = compose(config_name="tracker")
    ct(cfg)
    now = datetime.datetime.now()
    print('track DONE!', now)

START! 2026-09-07 09:47:39.307451
please check the configures:

path: ''
dataloader:
  division_detect: true
  start_frame: 0
  num_frame: 200
  itv: 1
  if_crop: false
  tile_num: 25
  pred_tile_num: 4
  overlap: 32
train:
  epochs: 100
  batch_size: 2
  lr: 0.0001
  train_load: false
  load: CP.pth
  val: false
track:
  max_movenment: 50
  load_track: false
  track_file: track_linear_solver.csv
  centroid_file: 2025-09-22-centroid.npy
  method: linear_solver
  run_num: 1
  last_itv: 20
  division: true
  post_pro: true
  min_length: 1
  prune_leaf: true
  merge: true
  jitter_thr: 0.6
  div_interval: 30
  nearest: false

------0th running start-------

queue size: 1/1 ------0th running end-------

------all running ended-------

2026-09-07 09:21:125-frame time cost: 1 s
track DONE! 2026-09-07 09:47:41.264451


## save as CTC format and a gif file

In [ ]:
mask_dir = abs_path(join('data', 'mask'))
track_dir = abs_path(join("result", "track_results.csv"))
centroid = abs_path(join("result", "centroid.npy"))

tracks = pd.read_csv(track_dir).to_numpy()
cnt = np.load(centroid, allow_pickle=True)
lineage_mask(mask_dir, tracks, cnt, save_path=r"result", res_path='RES1')
lineage_mask(mask_dir, tracks, cnt, save_path=r"result", res_path='RES1', saveRGB=True)

tracking results save as：track_mask.gif
